In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings 
warnings.filterwarnings("ignore", category=FutureWarning, module="seaborn")

# EDA

In [2]:
df = pd.read_csv("/kaggle/input/playground-series-s5e4/train.csv")
df_work = df.copy()
df.head()

/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1458: RuntimeWarning: invalid value encountered in greater
  has_large_values = (abs_vals > 1e6).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in less
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()
/usr/local/lib/python3.10/dist-packages/pandas/io/formats/format.py:1459: RuntimeWarning: invalid value encountered in greater
  has_small_values = ((abs_vals < 10 ** (-self.digits)) & (abs_vals > 0)).any()


,id,Podcast_Name,Episode_Title,Episode_Length_minutes,Genre,Host_Popularity_percentage,Publication_Day,Publication_Time,Guest_Popularity_percentage,Number_of_Ads,Episode_Sentiment,Listening_Time_minutes
0,0,Mystery Matters,Episode 98,NaN,True Crime,74.81,Thursday,Night,NaN,0.0,Positive,31.41998
1,1,Joke Junction,Episode 26,119.80,Comedy,66.95,Saturday,Afternoon,75.95,2.0,Negative,88.01241
2,2,Study Sessions,Episode 16,73.90,Education,69.97,Tuesday,Evening,8.97,0.0,Negative,44.92531
3,3,Digital Digest,Episode 45,67.17,Technology,57.22,Monday,Morning,78.70,2.0,Positive,46.27824
4,4,Mind & Body,Episode 86,110.51,Health,80.07,Monday,Afternoon,58.68,3.0,Neutral,75.61031


# Feature Engineering

In [3]:
def replace_episode_length(df):
    df_temp = df.copy()

    df_temp["Episode_Length_missing"] = pd.isna(df_temp["Episode_Length_minutes"])
    df_temp.loc[df_temp["Episode_Length_minutes"].isna(), "Episode_Length_minutes"] = df_temp["Episode_Length_minutes"].median()
    return df_temp

def replace_guest_popularity(df):
    df_temp = df.copy()

    df_temp["Guest_Popularity_percentage"] = pd.isna(df_temp["Guest_Popularity_percentage"])
    df_temp.loc[df_temp["Guest_Popularity_percentage"].isna(), "Guest_Popularity_percentage"] = df_temp["Guest_Popularity_percentage"].median()
    return df_temp

def replace_num_ads(df):
    df_temp = df.copy()

    df_temp["Number_of_Ads"] = pd.isna(df_temp["Number_of_Ads"])
    df_temp.loc[df_temp["Number_of_Ads"].isna(), "Number_of_Ads"] = 0
    return df_temp

def replace_outliers(df):
    df_temp = df.copy()

    df_temp.loc[df_temp["Episode_Length_minutes"] > 300, "Episode_Length_minutes"] = 120
    df_temp.loc[df_temp["Number_of_Ads"] > 10, "Number_of_Ads"] = 10
    df_temp.loc[df_temp["Host_Popularity_percentage"] < 20] = 20
    df_temp.loc[df_temp["Host_Popularity_percentage"] > 100] = 100
    df_temp.loc[df_temp["Guest_Popularity_percentage"] > 100] = 100
    
    return df_temp

def clean_data(df):
    df_temp = df.copy()
    df_temp = df_temp.drop(columns=["id"])

    # replacing outliers with max/min
    df_temp = replace_outliers(df_temp)
    df_temp = replace_guest_popularity(df_temp)
    df_temp = replace_episode_length(df_temp)
    df_temp = replace_num_ads(df_temp)

    df_temp["Episode_Title"] = df_temp.apply(lambda x: x["Episode_Title"] if not str(x["Episode_Title"]).isdigit() else "Episode " + str(x["Episode_Title"]), axis=1)
    
    return df_temp

In [4]:
from sklearn.preprocessing import StandardScaler

num_features = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage", 
                "Number_of_Ads"] # Removed Outcome var

# numerical features actually transformed
num_features_transform = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage"]

cat_features = ["Podcast_Name", "Genre", "Publication_Day", "Publication_Time", 
                "Episode_Sentiment"] # Episode Title removed since its now int

# This thing not used
features = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage", 
            "Number_of_Ads", "episode_number", "Podcast_Name", "Genre", "Publication_Day", 
            "Publication_Time", "Episode_Sentiment", "Episode_Length_missing"]

def make_cool_features(df):
    df_temp = df.copy()

    # Apply the function to create the new column
    df_temp['episode_number'] = df_temp['Episode_Title'].str.extract(r'Episode (\d+)').astype(int)
    df_temp = df_temp.drop(columns=["Episode_Title"])

    # standardizationfor numerical features
    scaler = StandardScaler()
    df_temp[num_features_transform] = scaler.fit_transform(df_temp[num_features_transform])

    # need this thing
    df_temp1 = df_temp.copy()

    # encoding categorical features
    df_temp = pd.get_dummies(df_temp, columns=cat_features)

    # log transformations
    for feature in num_features_transform:
        if (df_temp[feature] <= 0).any():
            shift_value = abs(df_temp[feature].min()) + 1
            df_temp[f'{feature}_log'] = np.log(df_temp[feature] + shift_value)
        else:
            df_temp[f'{feature}_log'] = np.log(df_temp[feature])

    # ah
    numerical_columns = df_temp.drop(columns=["Listening_Time_minutes"]).select_dtypes(include=['int64', 'float64']).columns.tolist()
    categorical_columns = df_temp.select_dtypes(include=['object', 'bool']).columns.tolist()

    # interaction temrs
    # interaction terms between numeric and cat
    for num_feat in numerical_columns:
        for cat_feat in categorical_columns:
            # interaction term
            df_temp[f"{num_feat}_{cat_feat}"] = df_temp[num_feat] * df_temp[cat_feat]

    # interaction terms between all numeric
    for i in range(len(numerical_columns) - 1):
        for j in range(i+1, len(numerical_columns)):
            df_temp[f"{numerical_columns[i]}_{numerical_columns[j]}"] = df_temp[numerical_columns[i]] * df_temp[numerical_columns[j]]

    continuous_num = ["Episode_Length_minutes", "Host_Popularity_percentage", "Guest_Popularity_percentage"]
    everything_else = ['Podcast_Name', 'Genre', 'Publication_Day', 'Publication_Time', 'Number_of_Ads', 'Episode_Sentiment', "episode_number"]

    # funky term
    # featureX - Groupby(featureY)[featureX].mean()
    for feat_X in continuous_num:
        for feat_y in everything_else:
            df_temp[f"cool_{feat_X}_{feat_y}"] = df_temp1[feat_X] - df_temp1.groupby(feat_y)[feat_X].transform('mean')
            
    
    return df_temp

In [5]:
from sklearn.linear_model import Ridge
from sklearn.linear_model import LinearRegression

# class CustomLinearRegression():
#     """
#     L2 Regularized linear regression model tolerating weird data in test
#     """
#     def __init__(self, alpha=1.0, fit_intercept=True, **kwargs)
#         self.model = Ridge(alpha=alpha, fit_intercept=fit_intercept, **kwargs)
#         self.feature_names = None
    
#     def fit(self, X, y):
#         # Store feature names for later use
#         self.feature_names = X.columns.tolist()
#         self.model = self.model.fit(X, y)
#         return self
    
#     def predict(self, X):
#         # Make normal predictions
#         predictions = self.model.predict(X)

#         # Create a mask of rows where all values are either 20 or 100
#         is_special = np.all(np.isin(X.values, [20, 100]), axis=1)
        
#         # Find indices of special rows
#         special_indices = np.where(is_special)[0]
        
#         # Only process the special rows
#         if 'Podcast_Name' in self.feature_names:
#             podcast_name_idx = self.feature_names.index('Podcast_Name')
#             for i in special_indices:
#                 # Access by column index instead of name since X.values is a numpy array
#                 if X.iloc[i, podcast_name_idx] == 100:
#                     predictions[i] = 100
#                 else:
#                     predictions[i] = 20
        
#         return predictions

In [6]:
%%time
df_cool = clean_data(df)
df_cool = make_cool_features(df_cool)
all_features = df_cool.drop(columns=["Listening_Time_minutes"]).columns
df_cool.info()

/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
/usr/local/lib/python3.10/dist-packages/pandas/core/computation/expressions.py:73: RuntimeWarning: invalid value encountered in greater
  return op(a, b)
<ipython-input-3-abb6532fb383>:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0.0' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  df_temp.loc[df_temp["Guest_Popularity_percentage"].isna(), "Guest_Popularity_percentage"] = df_temp["Guest_Popularity_percentage"].median()
<ipython-input-3-abb6532fb383>:19: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future ver

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Columns: 722 entries, Episode_Length_minutes to cool_Guest_Popularity_percentage_episode_number
dtypes: bool(83), float64(547), int64(84), object(8)
memory usage: 3.6+ GB
CPU times: user 19.4 s, sys: 5.32 s, total: 24.7 s
Wall time: 21 s


# Model Building

In [7]:
# imports
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.metrics import make_scorer

In [8]:
from sklearn.model_selection import train_test_split
df_baseline = df.copy()
categorical_columns = df_baseline.select_dtypes(include=['object']).columns
df_baseline = pd.get_dummies(df_baseline, columns=categorical_columns)
df_baseline["Episode_Length_minutes"].fillna(df_baseline["Episode_Length_minutes"].median(), inplace=True)
df_baseline["Guest_Popularity_percentage"].fillna(df_baseline["Guest_Popularity_percentage"].median(), inplace=True)
df_baseline["Number_of_Ads"].fillna(0, inplace=True)

X_train, X_val, y_train, y_val = train_test_split(df_baseline.drop(columns=['Listening_Time_minutes']), df_baseline["Listening_Time_minutes"], test_size = 0.3, random_state = 16)

<ipython-input-8-a68e8ee8d4f9>:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_baseline["Episode_Length_minutes"].fillna(df_baseline["Episode_Length_minutes"].median(), inplace=True)
<ipython-input-8-a68e8ee8d4f9>:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, wh

In [9]:
def rmse_loss(y_pred, y_actual):
    return np.sqrt(mean_squared_error(y_pred, y_actual))

rmse = make_scorer(rmse_loss, greater_is_better=False)

In [10]:
%%time
lr_baseline = Ridge().fit(X_train, y_train)

y_pred = lr_baseline.predict(X_val)
cool_score = rmse_loss(y_pred, y_val)
print(f"Baseline RMSE: {cool_score}")

/usr/local/lib/python3.10/dist-packages/sklearn/linear_model/_ridge.py:216: LinAlgWarning: Ill-conditioned matrix (rcond=4.06556e-17): result may not be accurate.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T


Baseline RMSE: 13.350904186171071
CPU times: user 1.84 s, sys: 541 ms, total: 2.38 s
Wall time: 1.84 s


## Feature Selection

In [11]:
from sklearn.feature_selection import SequentialFeatureSelector

In [12]:
%%time
X = df_cool.drop(columns=["Listening_Time_minutes"])
y = df_cool["Listening_Time_minutes"]
lr_unfitted = Ridge()
sfs = SequentialFeatureSelector(lr_unfitted, scoring=rmse, n_features_to_select=5)
sfs.fit(X, y)
selected_features = sfs.get_feature_names_out()
print(f"Selected Features: {selected_features}")

Selected Features: ['Episode_Length_minutes' 'episode_number_Episode_Length_missing'
 'Episode_Length_minutes_log_Episode_Sentiment_Positive'
 'Episode_Length_minutes_Host_Popularity_percentage'
 'Host_Popularity_percentage_Host_Popularity_percentage_log']
CPU times: user 26min 45s, sys: 2min 50s, total: 29min 35s
Wall time: 20min 37s


In [13]:
df_cool_selected = df_cool[selected_features]

X_train, X_val, y_train, y_val = train_test_split(df_cool[selected_features], df_cool["Listening_Time_minutes"], test_size=0.3, random_state=16)

lr_1 = Ridge().fit(X_train, y_train)

y_pred = lr_1.predict(X_val)
cool_score = rmse_loss(y_pred, y_val)
print(f"Feature Selection Only RMSE: {cool_score}")

Feature Selection Only RMSE: 13.474523319038726


In [14]:
# from sklearn.linear_model import Lasso

# lasso_model = Lasso(alpha=0.01, max_iter=10000, tol=0.0001)
# lasso_model.fit(X_train_cool, y_train_cool)

# cool_pred = lasso_model.predict(X_val_cool)
# cool_score = rmse(cool_pred, y_val_cool)
# print(f"Baseline RMSE: {cool_score}")

# cool_pred = lasso_model.predict(X_val_cool)
# cool_score = rmse(cool_pred, y_val_cool)
# print(f"Baseline RMSE: {cool_score}")

# Submission

In [15]:
# df_test = pd.read_csv("/kaggle/input/playground-series-s5e4/test.csv")

In [16]:
# df_test_clean = clean_test_set(df_test)
# df_test_cool = make_cool_features(df_test_clean)

In [17]:
# cols_to_drop = [col for col in df_test_cool.columns if '20' in col or '100' in col]
# df_test_cool = df_test_cool.drop(columns=cols_to_drop)

In [18]:
# submission = pd.read_csv("/kaggle/input/playground-series-s5e4/sample_submission.csv")
# submission["Listening_Time_minutes"] = final_model.predict(df_test_cool)

In [19]:
# submission.to_csv("submission.csv", index=False)
# submission.head()